In [2]:
# import packages
import platform
from pathlib import Path
import pandas as pd
import geopandas as gpd
import seaborn as sns
import calendar
import matplotlib.pyplot as plt
import pygris

In [46]:
# Define color palette
BLUE = "#3357BA"     
ORANGE = "#FF4D00"   
LIGHT_BLUE = "#8CB8CC"
LIGHT_ORANGE = "#F3C28E"
DARK = "#233240"
GREY = "#A7ADB4"

# Standard color palette
palette = [
    BLUE,
    ORANGE,
    LIGHT_BLUE,
    LIGHT_ORANGE,
    DARK,
    GREY
]

# Set Seaborn theme
sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette=palette,
    font="Arial",
    rc={
        # Figure
        "figure.figsize": (10, 6),
        "figure.dpi": 120,

        # Background
        "axes.facecolor": "white",
        "figure.facecolor": "white",

        # Grid
        "axes.grid": True,
        "grid.color": "#E6E9EC",
        "grid.linestyle": "-",
        "grid.linewidth": 0.7,

        # Axes
        "axes.edgecolor": "#D5DADF",
        "axes.linewidth": 0.8,
        "axes.labelcolor": DARK,
        "axes.titlecolor": DARK,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.labelsize": 11,

        # Ticks
        "xtick.color": DARK,
        "ytick.color": DARK,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,

        # Legend
        "legend.frameon": False,
        "legend.fontsize": 10,

        # Avoid scientific notation and offset notation
        "axes.formatter.useoffset": False,
        "axes.formatter.limits": (-100, 100),
    }
)

# Set default color cycle
plt.rcParams["axes.prop_cycle"] = plt.cycler(
    color=palette
)

In [26]:
# import cleaned fire dataset

project_fldr = Path.cwd().parents[1]

raw_fldr = project_fldr / 'data' / 'raw'
processed_fldr = project_fldr / 'data' / 'processed'

# read in CSV
fire_shp_25 = gpd.read_file(processed_fldr / "fire_shp_25/fire_shp_25.shp")
fire_shp_25.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 521 entries, 0 to 520
Data columns (total 23 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   YEAR_       521 non-null    float64 
 1   STATE       521 non-null    str     
 2   AGENCY      521 non-null    str     
 3   UNIT_ID     521 non-null    str     
 4   FIRE_NAME   521 non-null    str     
 5   INC_NUM     521 non-null    str     
 6   IRWINID     469 non-null    str     
 7   ALARM_DATE  521 non-null    str     
 8   CONT_DATE   514 non-null    str     
 9   C_METHOD    521 non-null    int32   
 10  CAUSE       521 non-null    int32   
 11  OBJECTIVE   521 non-null    float64 
 12  GIS_ACRES   521 non-null    float64 
 13  COMPLEX_ID  17 non-null     str     
 14  COMPLEX_NA  17 non-null     str     
 15  COMMENTS    120 non-null    str     
 16  FIRE_NUM    0 non-null      object  
 17  Shape_Leng  521 non-null    float64 
 18  Shape_Area  521 non-null    float64 
 19  

In [42]:
fire_shp_25['CAUSE'].value_counts()

CAUSE
14    202
1      98
2      63
10     41
7      30
11     26
9      26
5      22
4      10
8       1
15      1
6       1
Name: count, dtype: int64

In [36]:
# make alarm date column datetime
fire_shp_25['ALARM_DATE'] = pd.to_datetime(fire_shp_25['ALARM_DATE'])
fire_shp_25['CONT_DATE'] = pd.to_datetime(fire_shp_25['CONT_DATE'])

In [43]:
#map cause to code
cause_to_code = {
    "Lightning": 1,
    "Equipment Use": 2,
    "Smoking": 3,
    "Campfire": 4,
    "Debris Burning": 5,
    "Railroad": 6,
    "Arson": 7,
    "Children (Fire Play)": 8,
    "Miscellaneous / Other": 9,
    "Fireworks": 10,
    "Power Line": 11,
    "Structure": 12,
    "Ammunition / Explosives": 13,
    "Unknown (Human / Undetermined)": 14,
}

In [44]:
# Set the "selected" values
alarm_start_date = "2025-08-01" #alarm date dropdown range
alarm_end_date = "2025-12-31"
cont_start_date = "2025-08-01" #containment date dropdown range
cont_end_date = "2025-12-31"
cause = cause_to_code['Lightning']
size = 0

# Subset the data
fire_subset = fire_shp_25.loc[
    (fire_shp_25['ALARM_DATE'] >= alarm_start_date) &  # Start date
    (fire_shp_25['ALARM_DATE'] <= alarm_end_date) &    # End date
    (fire_shp_25['CONT_DATE'] >= cont_start_date) &    # Containment start date
    (fire_shp_25['CONT_DATE'] <= cont_end_date) &      # Containment end date
    (fire_shp_25['CAUSE'] == cause) &  
    (fire_shp_25['Shape_Area'] >= size),  
]

fire_subset.head()


,YEAR_,STATE,AGENCY,UNIT_ID,FIRE_NAME,INC_NUM,IRWINID,ALARM_DATE,CONT_DATE,C_METHOD,...,COMPLEX_ID,COMPLEX_NA,COMMENTS,FIRE_NUM,Shape_Leng,Shape_Area,GlobalID,Alarm Mont,Alarm Mo_1,geometry
2,2025.0,CA,USF,SNF,GARNET,00001684,{68B988A1-C190-4F0C-9BEA-A658D389D167},2025-08-24 00:00:00+00:00,2025-10-23 00:00:00+00:00,3,...,NaN,NaN,NaN,None,234717.355087,2.428892e+08,{DDD0B278-7655-4F20-829D-FEEEF1DC935F},8.0,Aug,"MULTIPOLYGON (((89617.826 -122364.58, 89614.97..."
3,2025.0,CA,CDF,FKU,SALT,00018739,{7F4A1474-6439-44F3-8F0D-083DCFD33639},2025-09-02 00:00:00+00:00,2025-09-17 00:00:00+00:00,7,...,NaN,NaN,Park incident clipped out per BEU,None,90080.360959,1.034115e+08,{6817A72C-90CF-4CD4-8F26-2B9622F1F140},9.0,Sep,"POLYGON ((-31970.799 -229822.278, -31947.464 -..."
8,2025.0,CA,USF,SRF,DILLON,00000715,{0140F0B6-2F32-4651-9008-E5AE2C38F547},2025-08-25 00:00:00+00:00,2025-11-06 00:00:00+00:00,7,...,NaN,NaN,NaN,None,76359.245741,4.827647e+07,{FF4F07C5-FB4C-45E2-8982-A51A936094BE},8.0,Aug,"MULTIPOLYGON (((-293706.391 397689.528, -29370..."
10,2025.0,CA,CDF,TCU,6-5,00012060,{8AA404C0-7A0B-46AA-8302-65E9E8EEC88B},2025-09-02 00:00:00+00:00,2025-09-13 00:00:00+00:00,3,...,{E13CBF1B-109F-4C0D-9C56-6B5E78DD74A9},TCU SEPTEMBER LIGHTNING,NaN,None,50111.368447,2.815802e+07,{B53D79E2-2269-41C2-8F91-9EC4ADBD5F78},9.0,Sep,"POLYGON ((-37080.218 -17859.032, -37087.475 -1..."
14,2025.0,CA,CDF,MMU,1-4,00020063,{28F644D1-4403-4056-B10B-1DEA33A4A3EF},2025-09-02 00:00:00+00:00,2025-09-04 00:00:00+00:00,3,...,{AB0DD6C1-4050-48A1-8D2E-C0864B8D1066},MMU LIGHTNING COMPLEX,CAL FIRE Intel Flight - N461DF (CI641),None,31962.188133,1.801282e+07,{0BFDACE2-96AA-4A12-8712-E0D5563F4118},9.0,Sep,"POLYGON ((-4869.786 -82221.141, -4841.946 -823..."
